In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.utils import shuffle
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv('/datasets/Churn.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB


In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


In [4]:
df.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,9091.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,4.997690,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.894723,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,2.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [5]:
df['Tenure'] = df['Tenure'].fillna(df['Tenure'].median())
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df = pd.get_dummies(df, drop_first=True)
df['Exited'].value_counts()

0    7963
1    2037
Name: Exited, dtype: int64

In [6]:
features = df.drop('Exited', axis=1)
target = df['Exited']
features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=0.4, random_state=12345)
features_valid, features_test, target_valid, target_test = train_test_split(features_valid, target_valid, test_size=0.5, random_state=12345)
print(features_train.shape)
print(features_valid.shape)
print(features_test.shape)

(6000, 11)
(2000, 11)
(2000, 11)


In [7]:
def evaluar_modelo(modelo, features_train, target_train, features_valid, target_valid):
    modelo.fit(features_train, target_train)
    predicciones = modelo.predict(features_valid)
    return f1_score(target_valid, predicciones)

In [8]:
#Prueba del modelo de Arbol de decisión
tree = evaluar_modelo(DecisionTreeClassifier(random_state=12345), features_train, target_train, features_valid, target_valid)
print('El valor f1 del arbol de decisión es:', tree)

El valor f1 del arbol de decisión es: 0.5036319612590799


In [9]:
#Prueba del modelo de regresión logistica
logistic = evaluar_modelo(LogisticRegression(random_state=12345), features_train, target_train, features_valid, target_valid)
print('El valor f1 de la regresión logistica es:', logistic)

El valor f1 de la regresión logistica es: 0.08786610878661089


In [10]:
#Prueba del modelo de bosque aleatorio
forest = evaluar_modelo( RandomForestClassifier(n_estimators=10, max_depth=10, random_state=12345), features_train, target_train, features_valid, target_valid)
print('El valor f1 del bosque aleatorio es:', forest)

El valor f1 del bosque aleatorio es: 0.5869894099848714


En esta parte calculé el valor F1 de los 3 modelos sin hacer algo con el desequilibrio de clases, ahora calcularé de nuevo los valores f1 pero aplicando 2 métodos para equilibrar las clases

In [11]:
tree = evaluar_modelo(DecisionTreeClassifier(random_state=12345, class_weight='balanced', max_depth=5), features_train, target_train, features_valid, target_valid)
print('El valor f1 del arbol de decisión es:', tree)

El valor f1 del arbol de decisión es: 0.5963791267305644


In [12]:
#Prueba del modelo de regresión logistica
logistic = evaluar_modelo(LogisticRegression(random_state=12345, class_weight='balanced'), features_train, target_train, features_valid, target_valid)
print('El valor f1 de la regresión logistica es:', logistic)

El valor f1 de la regresión logistica es: 0.44709626093874305


In [13]:
#Prueba del modelo de bosque aleatorio
forest = evaluar_modelo(RandomForestClassifier(n_estimators=10, max_depth=10, random_state=12345, class_weight='balanced'), features_train, target_train, features_valid, target_valid)
print('EL valor f1 del bosque aleatorio es:', forest)

EL valor f1 del bosque aleatorio es: 0.6038647342995168


In [14]:
features_zeros = features_train[target_train == 0]
features_ones = features_train[target_train == 1]
target_zeros = target_train[target_train == 0]
target_ones = target_train[target_train == 1]
repeat_features = pd.concat([features_ones] * 3)
repeat_target = pd.concat([target_ones] * 3)
features_upsampled = pd.concat([features_zeros, repeat_features])
target_upsampled = pd.concat([target_zeros, repeat_target])

features_upsampled, target_upsampled = shuffle(features_upsampled, target_upsampled, random_state=12345)

In [15]:
tree = evaluar_modelo(DecisionTreeClassifier(random_state=12345, max_depth=5), features_upsampled, target_upsampled, features_valid, target_valid)
print('El valor f1 del arbol de decisión es:', tree)

El valor f1 del arbol de decisión es: 0.586374695863747


In [16]:
#Prueba del modelo de regresión logistica
logistic = evaluar_modelo(LogisticRegression(random_state=12345), features_upsampled, target_upsampled, features_valid, target_valid)
print('El valor f1 de la regresión logistica es:', logistic)

El valor f1 de la regresión logistica es: 0.435129740518962


In [17]:

#Prueba del modelo de bosque aleatorio
forest = evaluar_modelo(RandomForestClassifier(n_estimators=10, max_depth=10, random_state=12345), features_upsampled, target_upsampled, features_valid, target_valid)
print('EL valor f1 del bosque aleatorio es:', forest)


EL valor f1 del bosque aleatorio es: 0.5979381443298969


Analizando los resultados de los 9 entrenamientos mi conclusión es que el mejor enfoque de entrenamiento para este caso es el equilibrio de clases mediante el ajuste de peso, y de los 3 modelos con este ajuste el mejor es el bosque aleatorio, que no solo da el F1 mas alto de 60% sino que supera el umbral del proyecto de 59%

In [18]:
# listas de valores a probar
n_estimators_list = [50, 100, 200]
max_depth_list = [None, 5, 10, 20]

mejor_f1 = 0
mejor_params = None

for n in n_estimators_list:
    for d in max_depth_list:
        model = RandomForestClassifier(
            n_estimators=n,
            max_depth=d,
            random_state=12345,
            class_weight='balanced'
        )
        model.fit(features_train, target_train)
        y_pred = model.predict(features_valid)
        f1 = f1_score(target_valid, y_pred)

        print(f"n_estimators={n}, max_depth={d}, F1={f1:.4f}")

        if f1 > mejor_f1:
            mejor_f1 = f1
            mejor_params = (n, d)

print(f"\n👉 Mejor combinación: n_estimators={mejor_params[0]}, max_depth={mejor_params[1]} con F1={mejor_f1:.4f}")


n_estimators=50, max_depth=None, F1=0.5662
n_estimators=50, max_depth=5, F1=0.6114
n_estimators=50, max_depth=10, F1=0.6124
n_estimators=50, max_depth=20, F1=0.5610
n_estimators=100, max_depth=None, F1=0.5639
n_estimators=100, max_depth=5, F1=0.6128
n_estimators=100, max_depth=10, F1=0.6235
n_estimators=100, max_depth=20, F1=0.5649
n_estimators=200, max_depth=None, F1=0.5652
n_estimators=200, max_depth=5, F1=0.6113
n_estimators=200, max_depth=10, F1=0.6220
n_estimators=200, max_depth=20, F1=0.5758

👉 Mejor combinación: n_estimators=100, max_depth=10 con F1=0.6235


In [19]:
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=12345, class_weight='balanced')
model.fit(features_train, target_train)
predict = model.predict(features_test)
f1 = f1_score(target_test, predict)
print('f1 del modelo final:', f1)
probabilidades = model.predict_proba(features_test)
auc = roc_auc_score(target_test, probabilidades[:, 1])
print('La metrica AUC-ROC es:', auc)

f1 del modelo final: 0.5973397823458283
La metrica AUC-ROC es: 0.8548085586092036


En este proyecto preparé los datos correctamente y detecté un desequilibrio de clases: cerca del 20% de los clientes abandonaban el banco, frente al 80% que permanecía. Para abordar este problema entrené tres modelos distintos y probé diferentes enfoques de corrección del desbalance.

El mejor desempeño lo alcanzó el Random Forest con class_weight='balanced', tras optimizar hiperparámetros mediante un bucle anidado de n_estimators y max_depth. Con esta configuración, el modelo logró un F1 final de 0.5973 y un AUC-ROC de 0.86, superando el objetivo planteado de F1 ≥ 0.59.

En conclusión, el modelo cumplió el criterio de calidad definido y mostró una buena capacidad de discriminación entre clientes que permanecen y los que abandonan. Esto confirma que el pipeline de preparación de datos, el manejo del desbalance y la selección de hiperparámetros fueron adecuados para resolver el problema.